# Notebook 01 - Setup do Ambiente e Preparacao dos Dados

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Instalacao e configuracao de dependencias
2. Download de datasets medicos (PubMedQA, MedQuAD)
3. Parse e processamento dos dados
4. Criacao de dados sinteticos de protocolos hospitalares
5. Preprocessamento e anonimizacao
6. Formatacao para fine-tuning (JSONL)

---
## 1. Instalacao de Dependencias

In [37]:
!pip install langgraph langchain langchain-ollama langchain-community faiss-cpu python-dotenv tiktoken datasets transformers torch accelerate peft bitsandbytes requests -q

---
## 2. Configuracao de Chaves de API

In [38]:
import os
from dotenv import load_dotenv

load_dotenv()

# Configuracao do Ollama
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

print("Ambiente configurado!")
print(f"Modelo Ollama: {OLLAMA_MODEL}")
print(f"URL Ollama: {OLLAMA_BASE_URL}")

Ambiente configurado!
Modelo Ollama: llama3.2
URL Ollama: http://localhost:11434


---
## 3. Download do PubMedQA

**Fonte:** https://github.com/pubmedqa/pubmedqa

O PubMedQA contem 1,000 perguntas e respostas medicas rotuladas por especialistas, baseadas em publicacoes do PubMed.

Cada registro contem:
- `QUESTION`: Pergunta de pesquisa clinica
- `LONG_CONTEXT`: Resumo do artigo PubMed
- `FINAL_DECISION`: Resposta (yes/no/maybe)

In [39]:
import json
import os
import requests

# Diretorio para armazenar datasets
datasets_dir = "../data/datasets"
os.makedirs(datasets_dir, exist_ok=True)

# =====================================================
# PubMedQA - Download do repositorio oficial
# =====================================================
print("="*60)
print("DOWNLOAD: PubMedQA Dataset")
print("="*60)
print("Fonte: https://github.com/pubmedqa/pubmedqa")
print("-"*60)

pubmedqa_dir = os.path.join(datasets_dir, "pubmedqa")
os.makedirs(pubmedqa_dir, exist_ok=True)

# URL do arquivo JSON (ori_pqal.json - dados originais rotulados)
pubmedqa_url = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"

pubmedqa_labeled = {}

try:
    print(f"Baixando ori_pqal.json (dados rotulados por especialistas)...")
    response = requests.get(pubmedqa_url, timeout=30, verify=False)
    response.raise_for_status()
    
    pubmedqa_labeled = json.loads(response.text)
    
    # Salvar localmente
    pubmedqa_file = os.path.join(pubmedqa_dir, "ori_pqal.json")
    with open(pubmedqa_file, 'w', encoding='utf-8') as f:
        json.dump(pubmedqa_labeled, f, indent=2, ensure_ascii=False)
    
    print(f"PubMedQA baixado com sucesso!")
    print(f"  Total de perguntas: {len(pubmedqa_labeled)}")
    print(f"  Arquivo: ori_pqal.json")
    
    # Mostrar exemplo
    primeiro_id = list(pubmedqa_labeled.keys())[0]
    exemplo = pubmedqa_labeled[primeiro_id]
    print(f"\n  Exemplo (PMID {primeiro_id}):")
    print(f"    Pergunta: {exemplo.get('QUESTION', 'N/A')[:100]}...")
    print(f"    Decisao: {exemplo.get('FINAL_DECISION', 'N/A')}")
    
except Exception as e:
    print(f"Erro ao baixar PubMedQA: {e}")
    print("Verifique sua conexao com a internet.")

DOWNLOAD: PubMedQA Dataset
Fonte: https://github.com/pubmedqa/pubmedqa
------------------------------------------------------------
Baixando ori_pqal.json (dados rotulados por especialistas)...
PubMedQA baixado com sucesso!
  Total de perguntas: 1000
  Arquivo: ori_pqal.json

  Exemplo (PMID 21645374):
    Pergunta: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
    Decisao: N/A


/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


---
## 4. Download do MedQuAD

**Fonte:** https://github.com/abachaa/MedQuAD

O MedQuAD contem 47,457 perguntas e respostas medicas criadas a partir de 12 websites do NIH (National Institutes of Health).

Os dados estao em formato XML com:
- `Question`: Pergunta medica
- `Answer`: Resposta detalhada
- Metadados: tipo de pergunta, foco, CUI UMLS

In [40]:
import zipfile
import io

# =====================================================
# MedQuAD - Download do repositorio oficial
# =====================================================
print("="*60)
print("DOWNLOAD: MedQuAD Dataset")
print("="*60)
print("Fonte: https://github.com/abachaa/MedQuAD")
print("Descricao: 47,457 QA pairs de 12 websites do NIH")
print("-"*60)

medquad_dir = os.path.join(datasets_dir, "MedQuAD")

# URL do repositorio ZIP
medquad_zip_url = "https://github.com/abachaa/MedQuAD/archive/refs/heads/master.zip"

try:
    if not os.path.exists(medquad_dir):
        print(f"Baixando repositorio MedQuAD...")
        response = requests.get(medquad_zip_url, timeout=60, verify=False)
        response.raise_for_status()
        
        # Extrair ZIP
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(datasets_dir)
        
        # Renomear diretorio
        extracted_dir = os.path.join(datasets_dir, "MedQuAD-master")
        if os.path.exists(extracted_dir):
            os.rename(extracted_dir, medquad_dir)
        
        print(f"MedQuAD baixado com sucesso!")
    else:
        print(f"MedQuAD ja existe em: {medquad_dir}")
    
    # Listar diretorios de QA
    import glob
    qa_dirs = [d for d in os.listdir(medquad_dir) if d.endswith('_QA')]
    print(f"\nDiretorios de QA encontrados: {len(qa_dirs)}")
    for d in sorted(qa_dirs):
        xml_files = glob.glob(os.path.join(medquad_dir, d, "*.xml"))
        print(f"  - {d}: {len(xml_files)} arquivos XML")
        
except Exception as e:
    print(f"Erro ao baixar MedQuAD: {e}")
    print("Verifique sua conexao com a internet.")

DOWNLOAD: MedQuAD Dataset
Fonte: https://github.com/abachaa/MedQuAD
Descricao: 47,457 QA pairs de 12 websites do NIH
------------------------------------------------------------
MedQuAD ja existe em: ../data/datasets/MedQuAD

Diretorios de QA encontrados: 11
  - 10_MPlus_ADAM_QA: 4366 arquivos XML
  - 11_MPlusDrugs_QA: 1312 arquivos XML
  - 12_MPlusHerbsSupplements_QA: 99 arquivos XML
  - 1_CancerGov_QA: 116 arquivos XML
  - 2_GARD_QA: 2685 arquivos XML
  - 3_GHR_QA: 1086 arquivos XML
  - 4_MPlus_Health_Topics_QA: 981 arquivos XML
  - 5_NIDDK_QA: 157 arquivos XML
  - 6_NINDS_QA: 277 arquivos XML
  - 7_SeniorHealth_QA: 48 arquivos XML
  - 9_CDC_QA: 59 arquivos XML


---
## 5. Parse dos Dados MedQuAD (XML -> JSON)

In [41]:
import xml.etree.ElementTree as ET
import glob

def parse_medquad_xml(xml_path):
    """Extrai pares pergunta-resposta de um arquivo XML do MedQuAD."""
    qa_pairs = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # MedQuAD tem Question e Answer como filhos diretos
        questions = root.findall('.//Question')
        answers = root.findall('.//Answer')
        
        for q, a in zip(questions, answers):
            q_text = q.text.strip() if q.text else ""
            a_text = a.text.strip() if a.text else ""
            
            if q_text and a_text and len(a_text) > 10:
                qa_pairs.append({
                    "question": q_text,
                    "answer": a_text,
                    "source": "MedQuAD",
                    "file": os.path.basename(xml_path)
                })
    except Exception as e:
        pass
    return qa_pairs

# Processar todos os XMLs do MedQuAD
medquad_all_qa = []

if os.path.exists(medquad_dir):
    print("Processando arquivos XML do MedQuAD...")
    
    qa_dirs_found = [d for d in os.listdir(medquad_dir) if d.endswith('_QA')]
    
    for qa_dir in sorted(qa_dirs_found):
        dir_path = os.path.join(medquad_dir, qa_dir)
        xml_files = glob.glob(os.path.join(dir_path, "*.xml"))
        
        dir_qa_count = 0
        for xml_file in xml_files:
            qa_pairs = parse_medquad_xml(xml_file)
            medquad_all_qa.extend(qa_pairs)
            dir_qa_count += len(qa_pairs)
        
        print(f"  {qa_dir}: {dir_qa_count} QA pairs")
    
    print(f"\nTotal de QA pairs do MedQuAD: {len(medquad_all_qa)}")
    
    # Salvar JSON processado
    medquad_json_file = os.path.join(datasets_dir, "medquad_processed.json")
    with open(medquad_json_file, 'w', encoding='utf-8') as f:
        json.dump(medquad_all_qa, f, indent=2, ensure_ascii=False)
    print(f"Salvo em: {medquad_json_file}")
    
    # Mostrar exemplo
    if medquad_all_qa:
        ex = medquad_all_qa[0]
        print(f"\nExemplo:")
        print(f"  Pergunta: {ex['question'][:100]}...")
        print(f"  Resposta: {ex['answer'][:100]}...")
else:
    print("Diretorio MedQuAD nao encontrado.")
    medquad_all_qa = []

Processando arquivos XML do MedQuAD...
  10_MPlus_ADAM_QA: 0 QA pairs
  11_MPlusDrugs_QA: 0 QA pairs
  12_MPlusHerbsSupplements_QA: 0 QA pairs
  1_CancerGov_QA: 729 QA pairs
  2_GARD_QA: 5389 QA pairs
  3_GHR_QA: 5430 QA pairs
  4_MPlus_Health_Topics_QA: 981 QA pairs
  5_NIDDK_QA: 1192 QA pairs
  6_NINDS_QA: 1088 QA pairs
  7_SeniorHealth_QA: 769 QA pairs
  9_CDC_QA: 269 QA pairs

Total de QA pairs do MedQuAD: 15847
Salvo em: ../data/datasets/medquad_processed.json

Exemplo:
  Pergunta: What is (are) Non-Small Cell Lung Cancer ?...
  Resposta: Key Points
                    - Non-small cell lung cancer is a disease in which malignant (cancer)...


---
## 6. Conversao dos Datasets Reais para Formato Fine-Tuning

In [42]:
# =====================================================
# PubMedQA -> Formato Fine-Tuning
# =====================================================
pubmedqa_ft_data = []

if pubmedqa_labeled:
    print(f"Convertendo {len(pubmedqa_labeled)} perguntas do PubMedQA...")
    
    for pmid, item in pubmedqa_labeled.items():
        question = item.get('QUESTION', '')
        decision = item.get('final_decision', '') or item.get('FINAL_DECISION', '')
        
        # Contexto do artigo
        context = item.get('LONG_CONTEXT', '') or item.get('ABSTRACT', '')
        if not context and 'CONTEXTS' in item:
            context = ' '.join(item['CONTEXTS'])
        
        # Resposta longa se disponivel
        long_answer = item.get('LONG_ANSWER', '')
        
        if question and (decision or long_answer):
            # Mapear respostas
            resposta_map = {
                'yes': 'Sim, com base na evidencia cientifica.',
                'no': 'Nao, a evidencia nao suporta essa afirmacao.',
                'maybe': 'E necessaria mais evidencia para concluir.'
            }
            resposta = long_answer if long_answer else resposta_map.get(decision.lower(), decision)
            instruction = 'Voce e um assistente medico. Responda a pergunta de pesquisa clinica com base na evidencia fornecida.'
            input_text = f"Contexto: {context[:500]}...\n\nPergunta: {question}"
            
            pubmedqa_ft_data.append({
                'instruction': instruction,
                'input': input_text,
                'output': resposta[:2000],
                'text': f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{resposta[:2000]}",
                'source': 'PubMedQA',
                'pmid': pmid
            })
    
    print(f"PubMedQA convertido: {len(pubmedqa_ft_data)} registros")
else:
    print("PubMedQA nao disponivel - Execute a celula de download anterior primeiro")

Convertendo 1000 perguntas do PubMedQA...
PubMedQA convertido: 1000 registros


In [43]:
# =====================================================
# MedQuAD -> Formato Fine-Tuning
# =====================================================
medquad_ft_data = []

if medquad_all_qa:
    print(f"Convertendo {len(medquad_all_qa)} QA pairs do MedQuAD...")
    
    for item in medquad_all_qa:
        question = item.get('question', '')
        answer = item.get('answer', '')
        
        if question and answer and len(answer) > 20:
            instruction = 'Voce e um assistente medico. Responda a pergunta sobre saude de forma clara e precisa.'
            medquad_ft_data.append({
                'instruction': instruction,
                'input': question,
                'output': answer[:2000],
                'text': f"### Instruction:\n{instruction}\n\n### Input:\n{question}\n\n### Response:\n{answer[:2000]}",
                'source': 'MedQuAD',
                'file': item.get('file', '')
            })
    
    print(f"MedQuAD convertido: {len(medquad_ft_data)} registros")
else:
    print("MedQuAD nao disponivel - Execute a celula de parse anterior primeiro")

Convertendo 15847 QA pairs do MedQuAD...
MedQuAD convertido: 15847 registros


---
## 7. Dados Sinteticos - Protocolos Hospitalares

In [44]:
# Protocolos medicos sinteticos do hospital
protocolos_hospitalares = [
    {
        "id": "PROTO-001",
        "titulo": "Protocolo de Pneumonia Hospitalar",
        "condicao": "Pneumonia Hospitalar",
        "condutas": [
            "Colher culturas antes do antibiotico",
            "Ceftriaxona 2g IV 24h + Azitromicina 500mg IV 24h",
            "Reavaliacao em 48-72 horas",
            "Ajuste conforme antibiograma",
            "Duracao minima: 7 dias"
        ],
        "exames": ["Raio-X torax", "Hemograma", "PCR", "Procalcitonina", "Gasometria"]
    },
    {
        "id": "PROTO-002",
        "titulo": "Protocolo de Sepse",
        "condicao": "Sepse",
        "condutas": [
            "Coletar lactato e hemoculturas antes do antibiotico",
            "Antibiotico de amplo espectro em 1 hora",
            "Resuscitacao: 30 mL/kg cristaloides",
            "Noradrenalina se PA < 90 mmHg",
            "Monitorizacao UTI"
        ],
        "exames": ["Lactato", "Hemoculturas", "Gasometria", "Hemograma"]
    },
    {
        "id": "PROTO-003",
        "titulo": "Protocolo de Insuficiencia Cardiaca",
        "condicao": "Insuficiencia Cardiaca",
        "condutas": [
            "Oxigenoterapia SaO2 > 90%",
            "Furosemida 40-80mg IV",
            "Nitroglicerina sublingual",
            "Restricao hidrica 1.5L/dia",
            "Monitorizacao continua"
        ],
        "exames": ["BNP", "Raio-X torax", "Ecocardiograma"]
    },
    {
        "id": "PROTO-004",
        "titulo": "Protocolo de AVC Isquemico",
        "condicao": "AVC Isquemico",
        "condutas": [
            "NIHSS imediata",
            "Tomografia urgente",
            "Trombolise ate 4.5h",
            "Trombectomia ate 24h",
            "Controle pressorico"
        ],
        "exames": ["Tomografia cranio", "RNM", "Hemograma"]
    }
]

# Converter protocolos para formato fine-tuning
protocolos_ft_data = []
for proto in protocolos_hospitalares:
    protocolos_ft_data.append({
        'instruction': f"Voce e um assistente medico especializado no hospital. Responda sobre {proto['condicao']}.",
        'input': f"Qual o protocolo para {proto['condicao']}?",
        'output': f"Protocolo {proto['titulo']}:\n\nCondutas:\n" + "\n".join(f"{i+1}. {c}" for i, c in enumerate(proto['condutas'])) + f"\n\nExames: {', '.join(proto['exames'])}",
        'source': 'Protocolo_Hospitalar'
    })

print(f"Protocolos convertidos: {len(protocolos_ft_data)} registros")

Protocolos convertidos: 4 registros


---
## 8. Funcoes de Anonimizacao (LGPD)

In [45]:
# Importar funcao de anonimizacao do modulo centralizado
from src import anonymize_text

# Criar alias em portugues para compatibilidade
anonimizar_texto = anonymize_text

print("Funcao de anonimizacao importada do modulo centralizado!")

Funcao de anonimizacao importada do modulo centralizado!


---
## 9. Consolidacao e Salvar Dataset Final

In [46]:
# Consolidar todos os dados
todos_dados = []
todos_dados.extend(pubmedqa_ft_data)
todos_dados.extend(medquad_ft_data)
todos_dados.extend(protocolos_ft_data)

print("="*60)
print("RESUMO DOS DADOS CONSOLIDADOS")
print("="*60)
print(f"PubMedQA:    {len(pubmedqa_ft_data):>6} registros")
print(f"MedQuAD:     {len(medquad_ft_data):>6} registros")
print(f"Protocolos:  {len(protocolos_ft_data):>6} registros")
print(f"-"*40)
print(f"TOTAL:       {len(todos_dados):>6} registros")
print("="*60)

# Salvar em JSONL
output_dir = "../data/dados_sinteticos"
os.makedirs(output_dir, exist_ok=True)

jsonl_file = os.path.join(output_dir, "dataset_fine_tuning.jsonl")
with open(jsonl_file, 'w', encoding='utf-8') as f:
    for item in todos_dados:
        f.write(json.dumps({
            'instruction': item['instruction'],
            'input': item['input'],
            'output': item['output']
        }, ensure_ascii=False) + '\n')

# Salvar em formato Alpaca
alpaca_file = os.path.join(output_dir, "dataset_alpaca.json")
alpaca_data = [{
    'instruction': item['instruction'],
    'input': item['input'],
    'output': item['output'],
    'text': f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"
} for item in todos_dados]

with open(alpaca_file, 'w', encoding='utf-8') as f:
    json.dump(alpaca_data, f, indent=2, ensure_ascii=False)

print(f"\nArquivos salvos:")
print(f"  JSONL: {jsonl_file}")
print(f"  Alpaca: {alpaca_file}")

RESUMO DOS DADOS CONSOLIDADOS
PubMedQA:      1000 registros
MedQuAD:      15847 registros
Protocolos:       4 registros
----------------------------------------
TOTAL:        16851 registros

Arquivos salvos:
  JSONL: ../data/dados_sinteticos/dataset_fine_tuning.jsonl
  Alpaca: ../data/dados_sinteticos/dataset_alpaca.json


In [47]:
# Analise do dataset
import statistics

tamanhos_in = [len(item['input']) for item in todos_dados]
tamanhos_out = [len(item['output']) for item in todos_dados]

print("\n=== ANALISE DO DATASET ===")
print(f"Total: {len(todos_dados)} registros")
print(f"\nInputs:")
print(f"  Media: {statistics.mean(tamanhos_in):.0f} chars")
print(f"  Mediana: {statistics.median(tamanhos_in):.0f} chars")
print(f"  Max: {max(tamanhos_in)} chars")
print(f"\nOutputs:")
print(f"  Media: {statistics.mean(tamanhos_out):.0f} chars")
print(f"  Mediana: {statistics.median(tamanhos_out):.0f} chars")
print(f"  Max: {max(tamanhos_out)} chars")

# Distribuicao por fonte
fontes = {}
for item in todos_dados:
    src = item.get('source', 'Outro')
    fontes[src] = fontes.get(src, 0) + 1

print(f"\nPor fonte:")
for src, count in sorted(fontes.items()):
    print(f"  {src}: {count}")


=== ANALISE DO DATASET ===
Total: 16851 registros

Inputs:
  Media: 85 chars
  Mediana: 50 chars
  Max: 738 chars

Outputs:
  Media: 959 chars
  Mediana: 808 chars
  Max: 2000 chars

Por fonte:
  MedQuAD: 15847
  Protocolo_Hospitalar: 4
  PubMedQA: 1000


---
## 9.5 Processamento dos Datasets Reais (PubMedQA e MedQuAD)

Agora vamos processar os datasets reais baixados e cria-los no formato de treino.

In [48]:
# Importar modulo de processamento
import sys
sys.path.append('..')
from src.dataset_processor import (
    process_pubmedqa,
    process_medquad,
    create_synthetic_protocols,
    create_synthetic_medical_qa,
    save_json,
    save_jsonl
)

print("Modulo de processamento importado com sucesso!")

Modulo de processamento importado com sucesso!


In [49]:
# Processar PubMedQA
pubmedqa_path = "../data/datasets/pubmedqa/ori_pqal.json"

if os.path.exists(pubmedqa_path):
    print("Processando PubMedQA...")
    pubmedqa_data = process_pubmedqa(pubmedqa_path, max_samples=1000)
    print(f"PubMedQA processado: {len(pubmedqa_data)} exemplos")
    print(f"Exemplo: {pubmedqa_data[0]['input'][:100]}...")
else:
    print(f"Arquivo nao encontrado: {pubmedqa_path}")
    print("Execute a celula de download anterior primeiro.")
    pubmedqa_data = []

Processando PubMedQA...
PubMedQA processado: 1000 exemplos
Exemplo: Contexto: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace p...


In [50]:
# Processar MedQuAD
medquad_dir = "../data/datasets/MedQuAD"

if os.path.exists(medquad_dir):
    print("Processando MedQuAD...")
    medquad_data = process_medquad(medquad_dir, max_samples=10000)
    print(f"MedQuAD processado: {len(medquad_data)} exemplos")
    if medquad_data:
        print(f"Exemplo: {medquad_data[0]['input'][:100]}...")
else:
    print(f"Diretorio nao encontrado: {medquad_dir}")
    print("Execute a celula de download anterior primeiro.")
    medquad_data = []

Processando MedQuAD...
MedQuAD processado: 10000 exemplos
Exemplo: What is (are) keratoderma with woolly hair ?...


In [51]:
# Criar dados sinteticos adicionais
print("Criando protocolos sinteticos...")
protocols = create_synthetic_protocols()
print(f"Protocolos criados: {len(protocols)}")

print("\nCriando QA sintetico...")
qa_sinteticos = create_synthetic_medical_qa()
print(f"QA sinteticos criados: {len(qa_sinteticos)}")

Criando protocolos sinteticos...
Protocolos criados: 8

Criando QA sintetico...
QA sinteticos criados: 10


In [52]:
# Combinar todos os datasets
todos_dados_finais = []

# Adicionar dados sinteticos
todos_dados_finais.extend(protocols)
todos_dados_finais.extend(qa_sinteticos)

# Adicionar dados reais
todos_dados_finais.extend(pubmedqa_data)
todos_dados_finais.extend(medquad_data)

print("="*60)
print("DATASET FINAL - RESUMO")
print("="*60)
print(f"Protocolos sinteticos: {len(protocols):>6}")
print(f"QA sinteticos:         {len(qa_sinteticos):>6}")
print(f"PubMedQA:              {len(pubmedqa_data):>6}")
print(f"MedQuAD:               {len(medquad_data):>6}")
print("-"*60)
print(f"TOTAL:                 {len(todos_dados_finais):>6}")
print("="*60)

DATASET FINAL - RESUMO
Protocolos sinteticos:      8
QA sinteticos:             10
PubMedQA:                1000
MedQuAD:                10000
------------------------------------------------------------
TOTAL:                  11018


In [53]:
# Salvar dataset final
output_dir = "../data"
os.makedirs(output_dir, exist_ok=True)

# Garantir que todos os registros tenham o campo 'text'
for item in todos_dados_finais:
    if 'text' not in item:
        instruction = item.get('instruction', '')
        inp = item.get('input', '')
        output = item.get('output', '')
        item['text'] = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{output}"

# Salvar em JSONL
jsonl_path = os.path.join(output_dir, "dataset_fine_tuning.jsonl")
save_jsonl(todos_dados_finais, jsonl_path)

# Salvar em formato Alpaca
alpaca_path = os.path.join(output_dir, "dataset_alpaca.json")
save_json(todos_dados_finais, alpaca_path)

print(f"\nDataset final salvo em:")
print(f"  JSONL: {jsonl_path}")
print(f"  Alpaca: {alpaca_path}")
print(f"\nTotal de exemplos para treino: {len(todos_dados_finais)}")


Dataset final salvo em:
  JSONL: ../data/dataset_fine_tuning.jsonl
  Alpaca: ../data/dataset_alpaca.json

Total de exemplos para treino: 11018


---
## 10. Resumo

### Datasets Processados:
| Dataset | Fonte | Registros | Descricao |
|---------|-------|-----------|----------|
| PubMedQA | NIH/PubMed | 300 | Perguntas de pesquisa clinica |
| MedQuAD | NIH/NLM | 300 | QA medicas do NIH |
| Protocolos | Sintetico | 8 | Protocolos hospitalares |
| QA Sintetico | Sintetico | 10 | Perguntas e respostas gerais |
| **TOTAL** | | **618** | Dataset completo para fine-tuning |

### Arquivos Gerados:
- `data/dataset_fine_tuning.jsonl`
- `data/dataset_alpaca.json`

### Proximo notebook:
Execute `02_fine_tuning_llm.ipynb` para treinar o modelo.

In [54]:
print(f"\n=== NOTEBOOK 01 CONCLUIDO ===")
print("\nProximo passo: Execute 02_fine_tuning_llm.ipynb")


=== NOTEBOOK 01 CONCLUIDO ===

Proximo passo: Execute 02_fine_tuning_llm.ipynb
